In [1]:
import pandas as pd

# あなたが作ったCSVを読み込む
df = pd.read_csv("npb_2025_main_fourth_batter_stats.csv")

# 数値化
num_cols = ["本塁打", "OBP", "SLG", "OPS", "打点", "試合", "打席", "打数", "安打", "四球", "死球", "三振", "併殺打"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# zスコア標準化
def zscore(series):
    return (series - series.mean()) / series.std(ddof=0)

df["z_HR"] = zscore(df["本塁打"])
df["z_OBP"] = zscore(df["OBP"])
df["z_SLG"] = zscore(df["SLG"])
df["z_OPS"] = zscore(df["OPS"])

# スコア作成
df["長打型スコア"] = 0.5 * df["z_HR"] + 0.3 * df["z_SLG"] + 0.2 * df["z_OPS"]
df["総合型スコア"] = 0.4 * df["z_OBP"] + 0.3 * df["z_SLG"] + 0.3 * df["z_OPS"]

# 分類
df["4番タイプ"] = df.apply(
    lambda row: "長打型" if row["長打型スコア"] > row["総合型スコア"] else "総合型",
    axis=1
)

# 見やすい形に整理
result = df[[
    "球団", "選手名", "4番出場回数",
    "本塁打", "OBP", "SLG", "OPS", "打点",
    "長打型スコア", "総合型スコア", "4番タイプ"
]].copy()

# スコアを見やすく丸める
result["長打型スコア"] = result["長打型スコア"].round(3)
result["総合型スコア"] = result["総合型スコア"].round(3)

# 保存
result.to_csv("npb_2025_main_fourth_batter_types.csv", index=False, encoding="utf-8-sig")

print(result.sort_values("長打型スコア", ascending=False))
print("\n保存完了: npb_2025_main_fourth_batter_types.csv")

        球団     選手名  4番出場回数  本塁打    OBP    SLG    OPS   打点  長打型スコア  総合型スコア  \
11      阪神   佐藤 輝明     126   40  0.345  0.579  0.924  102   2.156   0.979   
6       巨人   岡本 和真      66   15  0.416  0.598  1.014   49   0.830   1.999   
5       中日   細川 成也      75   20  0.367  0.489  0.856   58   0.443   0.672   
10      西武     ネビン     119   21  0.346  0.448  0.794   63   0.238   0.141   
9       楽天     ボイト      35   13  0.384  0.498  0.882   39   0.087   0.945   
0     DeNA    牧 秀悟      60   16  0.325  0.475  0.800   49   0.042   0.051   
2   ソフトバンク   山川 穂高      68   23  0.300  0.402  0.702   62   0.024  -0.738   
1    オリックス  杉本 裕太郎      74   16  0.332  0.426  0.758   53  -0.221  -0.179   
3     ヤクルト     オスナ      64   14  0.307  0.377  0.684   67  -0.665  -0.812   
8     日本ハム   野村 佑希      52    8  0.325  0.398  0.723   35  -0.890  -0.449   
7       広島   末包 昇大      63   11  0.296  0.373  0.669   62  -0.894  -0.977   
4      ロッテ   山本 大斗      47   11  0.262  0.338  0.600   33  -1.150  -1.633   

長打型：0.5z(HR) + 0.3z(SLG) + 0.2*z(OPS)

HRに0.5をおいた理由：長打型４番を1番わかりやすく表すのは、本塁打です。４番のイメージとして最も強いのが「一振りで点を取る力」なので、ここを最重要にした。
SLGに0.3をおいた理由：ただ、本塁打だけだと極端です。長打力は２塁打・３塁打も含めてみた方が良いので、長打率も入れています。本塁打そのものよりは少し軽く、でもかなり重要なので。
OPSに0.2をおいた理由：OPSは出塁率と長打率を合わせた、かなり便利な総合指標です。ただし、長打型スコアでは中心はあくまでHRとSLGなので、OPSは補助的に加える形にしています。
「本塁打を中心にしつつ、純粋な長打力と最低限の総合力も見る」
　　

総合型：0.4z(OBP) + 0.3z(SLG) + 0.3*z(OPS)

OBPに0.4をおいた理由：総合型で最も大事にしたいのは、まず出塁力です。４番でも、凡打が少なく、チャンスでつなげる打者は価値があります。そのため、総合型ではOBPを最も重くしています。
SLGに0.3をおいた理由：総合型でも４番である以上、長打力は必要です。ただし、長打型ほど「一発」に寄せたいわけではないので、OBPより少し軽い0.3です。
OPSに0.3をおいた理由：OPSは出塁と長打の両方を含むので、総合型の考え方にかなりあっている。そのためSLGと同程度の重みで入れています。
つまり総合型スコアは、「出塁できる」「長打もある」「総合的に打てる」